In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\TANVEER\Documents\Altamash\Funnel A_B testing\Data\SQL_Query_Results\session_metrics.csv"
)

np.random.seed(42)

df["experiment_group"] = np.random.choice(
    ["control", "treatment"],
    size=len(df),
    p=[0.5, 0.5]
)

df["experiment_conversion"] = df["purchased"]

In [9]:
df.head(10)


,user_session,user_id,session_start,session_duration_seconds,viewed,added_to_cart,purchased,revenue,product_count,experiment_group,experiment_conversion
0,18af673b-7fb9-4202-a66d-5c855bc0fd2d,619841242,2020-03-01 05:29:28,26,2,0,0,0.0,2,control,0
1,cc8a2df5-2541-453d-a465-b0b9f888e71c,325496098,2020-03-01 05:29:04,0,1,0,0,0.0,1,treatment,0
2,07c2bdcb-a37c-4444-85b3-c3e3089975e8,577096637,2020-03-01 05:28:26,13,0,2,0,0.0,2,treatment,0
3,fd47f5cd-6ed1-43f1-a9c6-df6f0f72538a,607916558,2020-03-01 05:28:24,0,1,0,0,0.0,1,treatment,0
4,268c0dae-3755-469d-bc7c-8e56f37888d9,607916558,2020-03-01 05:28:20,0,1,0,0,0.0,1,control,0
5,062d2f70-2039-4e9d-8530-ec12874e8cf0,622090052,2020-03-01 05:27:24,0,1,0,0,0.0,1,control,0
6,ab7d349f-db5d-4790-8ab1-31e5c894459d,622090043,2020-03-01 05:27:23,142,2,1,0,0.0,3,control,0
7,10316cd5-adef-486d-9915-c2ca7870c8e8,548866558,2020-03-01 05:27:21,12,0,3,0,0.0,1,treatment,0
8,a859392f-9169-4f8a-b761-0e106f666259,621899361,2020-03-01 05:26:29,0,1,0,0,0.0,1,treatment,0
9,b395bede-b451-41ca-bd41-7fb90b92813e,610682174,2020-03-01 05:26:22,0,1,0,0,0.0,1,treatment,0


In [7]:
df["experiment_conversion"].value_counts()

experiment_conversion
0      4380654
3        17082
2        16853
1        15855
4        15088
        ...   
202          1
192          1
156          1
193          1
282          1
Name: count, Length: 179, dtype: int64

In [10]:
baseline_rate = df.loc[
    df["experiment_group"] == "control",
    "purchased"
].mean()

target_lift = 0.03

treatment_probability = min(
    baseline_rate * (1 + target_lift),
    0.99
)

In [14]:
print(baseline_rate) 
print(treatment_probability)

0.2843032671625435
0.29283236517741984


In [17]:
treatment_users = (
    df["experiment_group"] == "treatment"
)

df.loc[treatment_users, "experiment_conversion"] = (
    np.random.binomial(
        1,
        treatment_probability,
        treatment_users.sum()
    )
)

# Verify actual observed conversion rates in both groups

actual_control_cr = df.loc[df["experiment_group"] == "control", "experiment_conversion"].mean()
actual_treatment_cr = df.loc[df["experiment_group"] == "treatment", "experiment_conversion"].mean()

print(f"Control Conversion Rate:   {actual_control_cr:.4f}")
print(f"Treatment Conversion Rate: {actual_treatment_cr:.4f}")
print(f"Observed Relative Lift:    {((actual_treatment_cr - actual_control_cr) / actual_control_cr):.2%}")

Control Conversion Rate:   0.2843
Treatment Conversion Rate: 0.2932
Observed Relative Lift:    3.12%


In [12]:
df.head(10)

,user_session,user_id,session_start,session_duration_seconds,viewed,added_to_cart,purchased,revenue,product_count,experiment_group,experiment_conversion
0,18af673b-7fb9-4202-a66d-5c855bc0fd2d,619841242,2020-03-01 05:29:28,26,2,0,0,0.0,2,control,0
1,cc8a2df5-2541-453d-a465-b0b9f888e71c,325496098,2020-03-01 05:29:04,0,1,0,0,0.0,1,treatment,0
2,07c2bdcb-a37c-4444-85b3-c3e3089975e8,577096637,2020-03-01 05:28:26,13,0,2,0,0.0,2,treatment,0
3,fd47f5cd-6ed1-43f1-a9c6-df6f0f72538a,607916558,2020-03-01 05:28:24,0,1,0,0,0.0,1,treatment,0
4,268c0dae-3755-469d-bc7c-8e56f37888d9,607916558,2020-03-01 05:28:20,0,1,0,0,0.0,1,control,0
5,062d2f70-2039-4e9d-8530-ec12874e8cf0,622090052,2020-03-01 05:27:24,0,1,0,0,0.0,1,control,0
6,ab7d349f-db5d-4790-8ab1-31e5c894459d,622090043,2020-03-01 05:27:23,142,2,1,0,0.0,3,control,0
7,10316cd5-adef-486d-9915-c2ca7870c8e8,548866558,2020-03-01 05:27:21,12,0,3,0,0.0,1,treatment,1
8,a859392f-9169-4f8a-b761-0e106f666259,621899361,2020-03-01 05:26:29,0,1,0,0,0.0,1,treatment,1
9,b395bede-b451-41ca-bd41-7fb90b92813e,610682174,2020-03-01 05:26:22,0,1,0,0,0.0,1,treatment,0


In [16]:
print(treatment_users)

0          False
1           True
2           True
3           True
4          False
           ...  
4536266     True
4536267     True
4536268     True
4536269     True
4536270     True
Name: experiment_group, Length: 4536271, dtype: bool


H0:
Traffic is split according to the expected ratio.

H1:
Traffic distribution differs from expected ratio.

### Sample ratio mismatch:

In [21]:
# Sample Ratio Mismatch (SRM) Test

from scipy.stats import chisquare

control_count = (df["experiment_group"] == "control").sum()
treatment_count = (df["experiment_group"] == "treatment").sum()

observed = [control_count, treatment_count]
total = sum(observed)
expected = [total * 0.5, total * 0.5]

stat, p_value = chisquare(observed, expected)
print(f"\nSRM Test:")
print(f"Chi-square Statistic: {stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.01:
    print("WARNING: SRM detected! Sample split differs significantly from 50/50.")
else:
    print("SRM check passed: Assignment ratio aligns with the expected 50/50 split.")


SRM Test:
Chi-square Statistic: 0.1696
P-value: 0.6805
SRM check passed: Assignment ratio aligns with the expected 50/50 split.


### Conversion Test:

In [22]:
from statsmodels.stats.proportion import proportions_ztest

control = df[
    df["experiment_group"] == "control"
]

treatment = df[
    df["experiment_group"] == "treatment"
]

conversions = [
    control["experiment_conversion"].sum(),
    treatment["experiment_conversion"].sum()
]

sample_sizes = [
    len(control),
    len(treatment)
]

z_stat, p_value = proportions_ztest(
    conversions,
    sample_sizes
)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: -20.862210444475604
P-value: 1.1808579785799296e-96


### Absolute and relative lift

In [24]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Separate groups
# --------------------------------------------------

control = df[df["experiment_group"] == "control"].copy()
treatment = df[df["experiment_group"] == "treatment"].copy()

# --------------------------------------------------
# 2. Calculate conversion rates
# --------------------------------------------------

control_conversion = control["experiment_conversion"].mean()
treatment_conversion = treatment["experiment_conversion"].mean()

# --------------------------------------------------
# 3. Absolute lift
# --------------------------------------------------

absolute_lift = (
    treatment_conversion - control_conversion
)

# --------------------------------------------------
# 4. Relative lift
# --------------------------------------------------

relative_lift = (
    absolute_lift / control_conversion
)

print("====================================")
print("EXPERIMENT RESULTS")
print("====================================")

print(f"Control Conversion:   {control_conversion:.4%}")
print(f"Treatment Conversion: {treatment_conversion:.4%}")

print(f"\nAbsolute Lift: {absolute_lift:.4%}")
print(f"Relative Lift: {relative_lift:.2%}")

EXPERIMENT RESULTS
Control Conversion:   28.4303%
Treatment Conversion: 29.3181%

Absolute Lift: 0.8878%
Relative Lift: 3.12%


### Confidence interval for conversion difference

Now we want to know:

How uncertain is our estimated treatment effect?

In [26]:
from statsmodels.stats.proportion import confint_proportions_2indep

# Number of conversions
control_conversions = control["experiment_conversion"].sum()
treatment_conversions = treatment["experiment_conversion"].sum()

# Number of users
control_users = len(control)
treatment_users = len(treatment)

# 95% CI for difference in conversion rates
low, high = confint_proportions_2indep(
    treatment_conversions,
    treatment_users,
    control_conversions,
    control_users,
    method="wald"
)

print("\n====================================")
print("95% CONFIDENCE INTERVAL")
print("====================================")

print(f"Conversion Difference: {absolute_lift:.4%}")
print(f"95% CI Lower: {low:.4%}")
print(f"95% CI Upper: {high:.4%}")


95% CONFIDENCE INTERVAL
Conversion Difference: 0.8878%
95% CI Lower: 0.8044%
95% CI Upper: 0.9712%


## Guardrail metrics

In [28]:
print(df.columns.tolist())
print(df.head())

['user_session', 'user_id', 'session_start', 'session_duration_seconds', 'viewed', 'added_to_cart', 'purchased', 'revenue', 'product_count', 'experiment_group', 'experiment_conversion']
                           user_session    user_id        session_start  \
0  18af673b-7fb9-4202-a66d-5c855bc0fd2d  619841242  2020-03-01 05:29:28   
1  cc8a2df5-2541-453d-a465-b0b9f888e71c  325496098  2020-03-01 05:29:04   
2  07c2bdcb-a37c-4444-85b3-c3e3089975e8  577096637  2020-03-01 05:28:26   
3  fd47f5cd-6ed1-43f1-a9c6-df6f0f72538a  607916558  2020-03-01 05:28:24   
4  268c0dae-3755-469d-bc7c-8e56f37888d9  607916558  2020-03-01 05:28:20   

   session_duration_seconds  viewed  added_to_cart  purchased  revenue  \
0                        26       2              0          0      0.0   
1                         0       1              0          0      0.0   
2                        13       0              2          0      0.0   
3                         0       1              0          0      

## Revenue per user/session

In [29]:
control_revenue = control["revenue"].mean()
treatment_revenue = treatment["revenue"].mean()

print("\n====================================")
print("REVENUE PER SESSION")
print("====================================")

print(f"Control:   ₹{control_revenue:.2f}")
print(f"Treatment: ₹{treatment_revenue:.2f}")


REVENUE PER SESSION
Control:   ₹1.40
Treatment: ₹1.40


In [31]:
## Relative difference in revenue per session

revenue_lift = (
    treatment_revenue - control_revenue
) / control_revenue

print(f"Revenue Lift: {revenue_lift:.2%}")

Revenue Lift: -0.25%


In [33]:
## Average Order Value

control_orders = control[
    control["experiment_conversion"] == 1
]

treatment_orders = treatment[
    treatment["experiment_conversion"] == 1
]

control_aov = control_orders["revenue"].mean()
treatment_aov = treatment_orders["revenue"].mean()

print("\n====================================")
print("AVERAGE ORDER VALUE")
print("====================================")

print(f"Control AOV:   ₹{control_aov:.2f}")
print(f"Treatment AOV: ₹{treatment_aov:.2f}")


AVERAGE ORDER VALUE
Control AOV:   ₹25.85
Treatment AOV: ₹1.39


In [34]:
aov_lift = (
    treatment_aov - control_aov
) / control_aov

print(f"AOV Relative Change: {aov_lift:.2%}")

AOV Relative Change: -94.61%


## Welch's t-test for revenue

In [35]:
from scipy.stats import ttest_ind

control_revenue = control["revenue"]
treatment_revenue = treatment["revenue"]

t_stat, revenue_p_value = ttest_ind(
    control_revenue,
    treatment_revenue,
    equal_var=False
)

print("\n====================================")
print("REVENUE GUARDRAIL TEST")
print("====================================")

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {revenue_p_value:.6f}")


REVENUE GUARDRAIL TEST
T-statistic: 0.3228
P-value: 0.746856


## Cart abandonment guardrail

In [36]:
control_cart_users = control[
    control["added_to_cart"] == 1
]

treatment_cart_users = treatment[
    treatment["added_to_cart"] == 1
]

In [37]:
control_abandonment = (
    1 -
    control_cart_users["experiment_conversion"].mean()
)

treatment_abandonment = (
    1 -
    treatment_cart_users["experiment_conversion"].mean()
)

print("\n====================================")
print("CART ABANDONMENT")
print("====================================")

print(
    f"Control:   {control_abandonment:.2%}"
)

print(
    f"Treatment: {treatment_abandonment:.2%}"
)


CART ABANDONMENT
Control:   66.35%
Treatment: 70.60%


## Product category analysis

In [39]:
raw = pd.read_csv(
    r"C:\Users\TANVEER\Documents\Altamash\Funnel A_B testing\Data\Raw_data\2020-Jan.csv"
)

raw["event_time"] = pd.to_datetime(
    raw["event_time"],
    errors="coerce"
)

raw["category_code"] = raw[
    "category_code"
].fillna("Unknown")

## Category level Funnel


In [46]:
category_sessions = raw[
    [
        "category_code",
        "user_session",
        "event_type"
    ]
].drop_duplicates()

category_funnel = (
    category_sessions
    .groupby("category_code")
    .agg(
        sessions=(
            "user_session",
            "nunique"
        ),
        purchase_events=(
            "event_type",
            lambda x: (
                x == "purchase"
            ).sum()
        )
    )
    .reset_index()
)

In [47]:
category_sessions.head()

,category_code,user_session,event_type
0,Unknown,4adb70bb-edbd-4981-b60f-a05bfd32683a,view
1,Unknown,c8c5205d-be43-4f1d-aa56-4828b8151c8a,view
2,Unknown,46a5010f-bd69-4fbe-a00d-bb17aa7b46f3,view
3,Unknown,546f6af3-a517-4752-a98b-80c4c5860711,view
4,Unknown,cff70ddf-529e-4b0c-a4fc-f43a749c0acb,view


## cleaner category conversion

In [51]:
category_session = (
    raw.groupby(
        ["category_code", "user_session"]
    )
    .agg(
        purchased=(
            "event_type",
            lambda x: int(
                "purchase" in x.values
            )
        )
    )
    .reset_index()
)

In [ ]:
category_results = (
    category_session
    .groupby("category_code")
    .agg(
        sessions=("user_session", "nunique"),
        purchases=("purchased", "sum")
    )
    .reset_index()
)

category_results["conversion_rate"] = (
    category_results["purchases"]
    / category_results["sessions"]
)

KeyError: "Label(s) ['purchased'] do not exist"